# 🔍 Demo: Analizador de Dependencias Mejorado

Este notebook demuestra todas las funcionalidades del analizador de dependencias mejorado para ingenieros:

1. **Anotaciones**: `@desc`, `@unit`, `@range`, `@category`, `@ref`, `@check`
2. **Categorización automática** por heurísticas
3. **Nodos de verificación** (asserts)
4. **Vista Inputs/Outputs**
5. **Modo Trace** (tabla paso a paso)
6. **Análisis de Sensibilidad**

---

## 1️⃣ Propiedades del Material con Anotaciones

Definimos propiedades del acero con decoradores completos:

In [ ]:
# @desc: Módulo de elasticidad del acero estructural# @unit: GPa# @category: material# @range: [190, 210]# @ref: AISC 360-22 Spec..E = 200# @desc: Resistencia a fluencia del acero A36# @unit: MPa# @category: material# @range: [235, 260]# @ref: ASTM A36.fy = 250# @desc: Resistencia última del acero A36# @unit: MPa# @category: material# @range: [400, 550]fu = 450print(f"E = {E} GPa, fy = {fy} MPa, fu = {fu} MPa")

## 2️⃣ Geometría de la Sección

Usamos `@category: geometry` para identificar dimensiones:

In [ ]:
# @desc: Ancho del ala
# @unit: mm
# @category: geometry
# @range: [100, 400]
bf = 200

# @desc: Espesor del ala
# @unit: mm
# @category: geometry
# @range: [8, 40]
tf = 15

# @desc: Altura total de la viga
# @unit: mm
# @category: geometry
# @range: [200, 1000]
d = 450

# @desc: Espesor del alma
# @unit: mm
# @category: geometry
tw = 10

# @desc: Altura del alma (sin alas)
# @unit: mm
h = d - 2 * tf

print(f"Sección: bf={bf}mm, tf={tf}mm, d={d}mm, tw={tw}mm, h={h}mm")

## 3️⃣ Propiedades de la Sección (sin @category - heurística)

Variables sin `@category` explícito serán categorizadas por heurística:

In [ ]:
# @desc: Área de la sección transversal
# @unit: mm²
A = 2 * bf * tf + (d - 2*tf) * tw

# @desc: Momento de inercia respecto al eje fuerte
# @unit: mm⁴
I_x = (bf * d**3) / 12 - ((bf - tw) * h**3) / 12

# @desc: Módulo de sección elástico
# @unit: mm³
S_x = I_x / (d / 2)

print(f"A = {A:.0f} mm², I_x = {I_x:.2e} mm⁴, S_x = {S_x:.2e} mm³")

## 4️⃣ Cargas Aplicadas

In [ ]:
# @desc: Momento flector máximo aplicado
# @unit: kN·m
# @category: load
# @range: [0, 500]
M_u = 180

# @desc: Fuerza axial de compresión
# @unit: kN
# @category: load
# @range: [0, 1000]
P_u = 250

# @desc: Longitud de la viga
# @unit: m
# @category: geometry
L = 6.0

print(f"M_u = {M_u} kN·m, P_u = {P_u} kN, L = {L} m")

## 5️⃣ Cálculo de Esfuerzos (Resultados)

In [ ]:
# @desc: Esfuerzo por flexión
# @unit: MPa
# @category: result
sigma_b = (M_u * 1e6) / S_x  # Convertir kN·m a N·mm

# @desc: Esfuerzo por carga axial
# @unit: MPa
# @category: result
sigma_a = (P_u * 1e3) / A  # Convertir kN a N

# @desc: Esfuerzo total combinado
# @unit: MPa
# @category: result
# @range: [0, 250]
sigma_total = sigma_b + sigma_a

print(f"σ_b = {sigma_b:.1f} MPa, σ_a = {sigma_a:.1f} MPa")
print(f"σ_total = {sigma_total:.1f} MPa (límite: {fy} MPa)")

## 6️⃣ Factor de Seguridad

In [ ]:
# @desc: Factor de seguridad contra fluencia
# @category: factor
# @range: [1.5, 5.0]
FS = fy / sigma_total

# @desc: Ratio de demanda/capacidad
# @category: factor
# @range: [0, 1.0]
DCR = sigma_total / fy

print(f"Factor de Seguridad: {FS:.2f}")
print(f"DCR (Demand/Capacity): {DCR:.2%}")

## 7️⃣ Verificaciones con `@check`

Las verificaciones con `assert` aparecerán como nodos especiales:

In [ ]:
# @check: Verificación de resistencia a fluencia
assert sigma_total <= fy, f"FALLA: σ_total ({sigma_total:.1f}) > fy ({fy})"

# @check: Factor de seguridad mínimo
assert FS >= 1.5, f"FS = {FS:.2f} < 1.5 mínimo requerido"

# @check: Ratio DCR admisible
assert DCR <= 1.0, f"DCR = {DCR:.2%} supera el 100%"

print("✅ Todas las verificaciones PASARON")

## 8️⃣ Referencia Normativa Completa

In [ ]:
# @desc: Factor de resistencia LRFD
# @category: factor
# @ref: AISC 360-22 F1.1
phi_b = 0.90

# @desc: Resistencia nominal a flexión
# @unit: kN·m
# @category: result
# @ref: AISC 360-22 F2.1
M_n = phi_b * fy * S_x / 1e6  # En kN·m

# @desc: Resistencia disponible a flexión
# @unit: kN·m
# @category: result
# @ref: AISC 360-22 F1
phi_M_n = phi_b * M_n

print(f"φM_n = {phi_M_n:.1f} kN·m (disponible)")
print(f"M_u  = {M_u:.1f} kN·m (demanda)")
print(f"Ratio: {M_u/phi_M_n:.2%}")

---

## 📊 Cómo Usar el Analizador

1. **Selecciona cualquier variable** (ej: `sigma_total`)
2. **Clic derecho → Ver Dependencias** (o Ctrl+Shift+D)
3. El grafo mostrará:
   - 📥 **Inputs** (izquierda): E, fy, bf, tf, M_u, etc.
   - 📤 **Outputs** (derecha): sigma_total, FS, DCR
   - ✅ **Checks**: Nodos de verificación
4. **Botones disponibles:**
   - 📊 **Trace**: Ver tabla paso a paso del cálculo
   - 📈 **Sensibilidad**: Sliders para ajustar inputs

---

### Sintaxis Completa de Anotaciones

```python
# @desc: Descripción semántica de la variable
# @unit: Unidad física (kN, mm, MPa, etc.)
# @range: [valor_min, valor_max]
# @category: material | geometry | load | result | factor
# @ref: Referencia normativa (ACI 318, AISC 360, etc.)
# @check: Marca como nodo de verificación
```